# 1. Multi-Stage Attack Investigation

Real attacks aren't single events — they're chains: phishing → credential theft → lateral movement → data exfiltration. The SC-200 exam tests your ability to **trace the full kill chain** across data sources.

## The attack we'll investigate

The log generator injected a realistic multi-stage attack:

```
Stage 1: Phishing email delivered to alice@contoso.com
Stage 2: 15 failed + 1 successful sign-in from Moscow (brute force)
Stage 3: psexec.exe and mimikatz.exe on alice's laptop (lateral movement)
Stage 4: Large data uploads to known-bad IPs (exfiltration)
```

Your job: **find all four stages using queries**.

In [ ]:
import httpx, json
from collections import Counter

SIEM = 'http://localhost:8000'

print('=== Multi-Stage Attack Investigation ===\n')
print('Let\'s trace the attack through each stage.\n')

# Stage 1: Find the phishing email
print('--- Stage 1: Initial Access (Phishing) ---')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'EmailEvents',
    'filter': {'ThreatTypes': 'Phish'},
    'limit': 10,
})
phish = r.json()['results']
print(f'Found {len(phish)} phishing emails:')
for p in phish:
    delivered = '📬 DELIVERED' if p['DeliveryAction'] == 'Delivered' else '🚫 Blocked'
    print(f'  {delivered} From: {p["SenderFromAddress"]} → {p["RecipientEmailAddress"]}')
    print(f'           Subject: {p["Subject"]}')

In [ ]:
# Stage 2: Credential Access (Brute force)
print('\n--- Stage 2: Credential Access (Brute Force) ---')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'filter': {'UserPrincipalName': 'alice@contoso.com'},
    'limit': 50,
})
alice_signins = r.json()['results']

failures = [s for s in alice_signins if s['ResultType'] == 'Failure']
successes = [s for s in alice_signins if s['ResultType'] == 'Success']
print(f'Total sign-ins for alice: {len(alice_signins)}')
print(f'  Failures: {len(failures)}')
print(f'  Successes: {len(successes)}')

# Check IPs
ip_counts = Counter(s['IPAddress'] for s in failures)
print(f'\nFailure source IPs:')
for ip, count in ip_counts.most_common():
    print(f'  {ip}: {count} failures')

# Check locations
locations = Counter(s['Location'] for s in alice_signins)
print(f'\nSign-in locations:')
for loc, count in locations.most_common():
    suspicious = ' ⚠️ SUSPICIOUS' if loc in ('Moscow', 'Beijing', 'Anonymous Proxy') else ''
    print(f'  {loc}: {count}{suspicious}')

In [ ]:
# Stage 3: Lateral Movement
print('\n--- Stage 3: Lateral Movement ---')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'DeviceEvents',
    'filter': {'AccountName': 'alice'},
    'limit': 30,
})
alice_endpoint = r.json()['results']

suspicious_tools = {'mimikatz.exe', 'psexec.exe', 'cmd.exe', 'powershell.exe', 'certutil.exe'}
print(f'Endpoint events for alice: {len(alice_endpoint)}')
print(f'\nSuspicious processes:')
for event in alice_endpoint:
    if event['FileName'] in suspicious_tools:
        print(f'  🔴 {event["DeviceName"]}: {event["FileName"]} ({event["ActionType"]})')
        print(f'       Path: {event["FolderPath"]}')

# Which devices did alice's activity touch?
devices = Counter(e['DeviceName'] for e in alice_endpoint)
print(f'\nDevices accessed by alice:')
for device, count in devices.most_common():
    print(f'  {device}: {count} events')

In [ ]:
# Stage 4: Data Exfiltration
print('\n--- Stage 4: Exfiltration ---')

known_bad_ips = ['185.220.101.42', '45.33.32.156', '198.51.100.99']
for bad_ip in known_bad_ips:
    r = httpx.post(f'{SIEM}/query', json={
        'table_name': 'AzureFirewall',
        'filter': {'DestinationIP': bad_ip},
        'limit': 20,
    })
    results = r.json()['results']
    if results:
        print(f'⚠️  {len(results)} connections to known-bad IP {bad_ip}:')
        sources = Counter(e['SourceIP'] for e in results)
        for src, count in sources.most_common():
            print(f'    Source: {src} ({count} connections)')

print('\n--- Attack Timeline Summary ---')
print('1. 📧 Phishing email from m1crosoft-support.com delivered to alice')
print('2. 🔑 15 failed sign-ins from Moscow → 1 success (brute force)')
print('3. ↔️  psexec + mimikatz on laptop-alice → spread to vm-app-01, vm-db-01')
print('4. 📤 10 outbound connections to known-bad IPs (data exfiltration)')
print('\n✅ Full kill chain traced across 4 data sources!')

## SC-200: How this maps to real investigation

| What we did | Real Sentinel/XDR equivalent |
|------------|-----------------------------|
| Query EmailEvents for phishing | Defender for Office 365 → Email entity page |
| Query SigninLogs for brute force | Entra ID → Sign-in logs blade |
| Query DeviceEvents for lateral movement | Defender for Endpoint → Device timeline |
| Query AzureFirewall for exfiltration | Sentinel → KQL hunting |
| Correlate across sources | Defender XDR → Incident graph |

### Exam tip: investigation tools

| Tool | What it shows |
|------|--------------|
| **Incident graph** | Visual map of entities and their relationships |
| **Device timeline** | Chronological events on a specific device |
| **User page** | All activity for a user across products |
| **Advanced Hunting** | Raw KQL queries across all tables |
| **Live response** | Remote shell on a device for forensics |